In [8]:
import os
import random
import json
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
   torch.cuda.manual_seed_all(42)

Initially, we used only the OpenSubtitles data from OPUS, but the results were unsatisfactory. We then experimented with a mix of OpenSubtitles and CCMatrix data, but the performance still didn’t improve. Ultimately, we decided to rely solely on data from the OPUS-100 dataset.

As for the model configuration, the current setup was derived through iterative experimentation. We tested various combinations of layers, attention heads, d_model, and d_ff values, and the current configuration proved to be the most effective. For the first iteration, we followed the setup provided by the university in the Practice Notebook.

In [9]:
class Config:
    """Training configuration"""
    # Data
    data_dir = '../../data/OPUS'
    en_file = 'opus100_en_clean.txt'
    pl_file = 'opus100_pl_clean.txt'
    max_samples_opus = 330000
    max_samples_ccmatrix = 0
    train_split = 0.9

    # Vocabulary
    max_vocab_size = 50000
    max_seq_length = 64

    # Model
    d_model = 512
    n_heads = 8
    n_layers = 6
    d_ff = 2048
    dropout = 0.1

    # Training
    batch_size = 32
    gradient_accumulation_steps = 4
    epochs = 50
    learning_rate = 0.0003
    label_smoothing = 0.05
    grad_clip = 1.0
    early_stopping_patience = 5
    lr_reduce_patience = 5
    lr_reduce_factor = 0.5

    # Saving
    checkpoint_dir = 'checkpoints'
    log_file = 'training.log'

In [10]:
config = Config()
os.makedirs(config.checkpoint_dir, exist_ok=True)

In [11]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [12]:
# torch debug
print(f"Torch version: {torch.__version__}")
print(f"Cuda available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Cuda version: {torch.version.cuda}")
    print(f"CuDNN version: {torch.backends.cudnn.version()}")
    print(f"GPU count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")


Torch version: 2.8.0+cu129
Cuda available: True
Cuda version: 12.9
CuDNN version: 91002
GPU count: 1
  GPU 0: NVIDIA GeForce RTX 5080


In [13]:
def load_mixed_dataset(config):
    """Load and mix multiple data sources efficiently"""
    import time
    start_time = time.time()

    # 1. Load OpenSubtitles (main dataset)
    print("Loading OpenSubtitles data...")
    en_path = os.path.join(config.data_dir, config.en_file)
    pl_path = os.path.join(config.data_dir, config.pl_file)

    opus_en_len = 0
    opus_pl_len = 0

    # Use simple file reading for speed (avoid chardet for large files)
    with open(en_path, 'r', encoding='utf-8') as f:
        opus_en = [line.strip() for line in f if line.strip()][:config.max_samples_opus]
        opus_en_len = len(opus_en)

    with open(pl_path, 'r', encoding='utf-8') as f:
        opus_pl = [line.strip() for line in f if line.strip()][:config.max_samples_opus]
        opus_pl_len = len(opus_pl)

    assert opus_en_len == opus_pl_len, "Mismatched number of sentences in OpenSubtitles"
    print(f"Loaded {opus_en_len} sentence pairs from OpenSubtitles")

    # 4. Combine all sources
    all_en = opus_en
    all_pl = opus_pl

    # 5. Filter by length ratio
    filtered_pairs = []
    for en, pl in zip(all_en, all_pl):
        if en and pl:
            en_len = len(en.split())
            pl_len = len(pl.split())
            if 3 <= en_len < 30 and 3 <= pl_len < 30:
                ratio = en_len / max(pl_len, 1)
                if 0.5 < ratio < 2.0:
                    filtered_pairs.append((en, pl))

    # 6. Shuffle
    random.shuffle(filtered_pairs)
    all_en, all_pl = zip(*filtered_pairs) if filtered_pairs else ([], [])

    print(f"Total dataset size after filtering: {len(all_en)} pairs")
    print(f"Loading time: {time.time() - start_time:.2f} seconds")

    # Show samples
    print("\nSample pairs:")
    for i in range(min(3, len(all_en))):
        print(f"  EN: {all_en[i]}")
        print(f"  PL: {all_pl[i]}\n")

    return list(all_en), list(all_pl)

In [14]:
en_sentences, pl_sentences = load_mixed_dataset(config)

Loading OpenSubtitles data...
Loaded 330000 sentence pairs from OpenSubtitles
Total dataset size after filtering: 283215 pairs
Loading time: 0.63 seconds

Sample pairs:
  EN: Peter, embrace it. Yeah.
  PL: Peter... pogódź się z tym.

  EN: I didn't have much of a say in the matter. I got caught stealing, they gave me a choice:
  PL: Ukradłem samochód i sędzia dał mi wybór - pół roku więzienia albo 3 lata w wojsku.

  EN: Please say it's enough.
  PL: Proszę, powiedz, że wystarczy.



We switched to a SentencePiece tokenizer to better handle subwords and unknown words. Previously, we used a basic word-level tokenizer, but it led to a high number of out-of-vocabulary tokens and poor translation quality.

In [15]:
import sentencepiece as spm

def train_sentencepiece(sentences, model_prefix, vocab_size=16000):
    """Train SentencePiece model"""
    # Write sentences to temp file
    with open(f'{model_prefix}.txt', 'w', encoding='utf-8') as f:
        for sent in sentences:
            f.write(sent + '\n')

    # Train
    spm.SentencePieceTrainer.train(
        input=f'{model_prefix}.txt',
        model_prefix=model_prefix,
        vocab_size=vocab_size,
        character_coverage=0.9995,
        model_type='bpe',  # or 'unigram'
        pad_id=0,
        unk_id=1,
        bos_id=2,
        eos_id=3
    )

    return spm.SentencePieceProcessor(model_file=f'{model_prefix}.model')

# Train both models
if torch.cuda.get_device_name(0) == 'NVIDIA GeForce RTX 5080':
    print("training SentencePiece models...")
    pl_sp = train_sentencepiece(pl_sentences, 'pl_sp', vocab_size=16000)
    en_sp = train_sentencepiece(en_sentences, 'en_sp', vocab_size=16000)

training SentencePiece models...


In [16]:
pl_sp = spm.SentencePieceProcessor(model_file='pl_sp.model')
en_sp = spm.SentencePieceProcessor(model_file='en_sp.model')

In [17]:
class SPDataset(Dataset):
    def __init__(self, src_sentences, tgt_sentences, src_sp, tgt_sp, max_length=32):
        self.src_sp = src_sp
        self.tgt_sp = tgt_sp
        self.max_length = max_length

        # Encode all sentences
        self.src_tokens = [src_sp.encode(s, out_type=int, add_bos=True, add_eos=True)[:max_length]
                          for s in src_sentences]
        self.tgt_tokens = [tgt_sp.encode(s, out_type=int, add_bos=True, add_eos=True)[:max_length]
                          for s in tgt_sentences]

        # Pad
        self.src_tokens = [t + [0]*(max_length-len(t)) for t in self.src_tokens]
        self.tgt_tokens = [t + [0]*(max_length-len(t)) for t in self.tgt_tokens]

    def __len__(self):
        return len(self.src_tokens)

    def __getitem__(self, idx):
        return torch.tensor(self.src_tokens[idx]), torch.tensor(self.tgt_tokens[idx])

In [18]:
train_size = int(config.train_split * len(en_sentences))

train_dataset = SPDataset(
    pl_sentences[:train_size],
    en_sentences[:train_size],
    pl_sp, en_sp,
    config.max_seq_length
)

val_dataset = SPDataset(
    pl_sentences[train_size:],
    en_sentences[train_size:],
    pl_sp, en_sp,
    config.max_seq_length
)

In [19]:
train_loader = DataLoader(
        train_dataset,
        batch_size=config.batch_size,
        shuffle=True,
        pin_memory=True
    )

val_loader = DataLoader(
        val_dataset,
        batch_size=config.batch_size,
        shuffle=False,
        pin_memory=True
    )

print("\nDataset sizes:")
print(f"  Training: {len(train_dataset)}")
print(f"  Validation: {len(val_dataset)}")


Dataset sizes:
  Training: 254893
  Validation: 28322


In [20]:
class PositionalEncoding(nn.Module):
    """Positional encoding for transformer"""

    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() *
                           (-np.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)

        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


class TranslationTransformer(nn.Module):
    def __init__(self, src_vocab_size: int, tgt_vocab_size: int,
                 d_model: int = 256, n_heads: int = 8, n_layers: int = 4,
                 d_ff: int = 1024, dropout: float = 0.1, max_len: int = 100):
        super().__init__()

        self.d_model = d_model
        self.src_pad_idx = 0  # SentencePiece always uses 0 for padding
        self.tgt_pad_idx = 0

        # Embeddings
        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model)

        # Positional encoding
        self.pos_encoder = PositionalEncoding(d_model, dropout, max_len)

        # Transformer
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=n_heads,
            num_encoder_layers=n_layers,
            num_decoder_layers=n_layers,
            dim_feedforward=d_ff,
            dropout=dropout,
            batch_first=True
        )

        # Output projection
        self.fc_out = nn.Linear(d_model, tgt_vocab_size)

    def forward(self, src, tgt):
        # Keep as bool - PyTorch handles this
        src_padding_mask = (src == self.src_pad_idx)
        tgt_padding_mask = (tgt == self.tgt_pad_idx)

        # Embeddings
        src_emb = self.src_embedding(src) * np.sqrt(self.d_model)
        tgt_emb = self.tgt_embedding(tgt) * np.sqrt(self.d_model)

        src_emb = self.pos_encoder(src_emb)
        tgt_emb = self.pos_encoder(tgt_emb)

        # Causal mask
        tgt_seq_len = tgt.size(1)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt_seq_len).to(tgt.device)

        # Forward - PyTorch will handle the type conversion
        output = self.transformer(
            src_emb, tgt_emb,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_padding_mask,
            tgt_key_padding_mask=tgt_padding_mask
        )

        return self.fc_out(output)

In [21]:
model = TranslationTransformer(
    src_vocab_size=len(pl_sp),  # Use len() on SentencePiece model
    tgt_vocab_size=len(en_sp),
    d_model=config.d_model,
    n_heads=config.n_heads,
    n_layers=config.n_layers,
    d_ff=config.d_ff,
    dropout=config.dropout,
    max_len=config.max_seq_length
).to(device)

print(f"\nModel parameters: {sum(p.numel() for p in model.parameters()):,}")


Model parameters: 68,732,544


In [15]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.001, mode='min'):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.mode = mode

    def __call__(self, score):
        if self.best_score is None:
            self.best_score = score
        elif (self.mode == 'min' and score > self.best_score - self.min_delta) or \
             (self.mode == 'max' and score < self.best_score + self.min_delta):
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.counter = 0
        return self.early_stop

In [16]:
def train_epoch(model, dataloader, criterion, optimizer, scheduler, device, config):
    model.train()
    total_loss = 0
    optimizer.zero_grad()  # Zero once at start

    progress_bar = tqdm(dataloader, desc="Training", leave=False)
    for step, (src, tgt) in enumerate(progress_bar):
        src = src.to(device)
        tgt = tgt.to(device)

        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]

        output = model(src, tgt_input)
        output = output.reshape(-1, output.size(-1))
        tgt_output = tgt_output.reshape(-1)

        loss = criterion(output, tgt_output)
        loss = loss / config.gradient_accumulation_steps  # Scale loss

        loss.backward()

        # Only update every N steps
        if (step + 1) % config.gradient_accumulation_steps == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * config.gradient_accumulation_steps
        progress_bar.set_postfix({'loss': f'{loss.item() * config.gradient_accumulation_steps:.4f}'})

    return total_loss / len(dataloader)

In [17]:
criterion = nn.CrossEntropyLoss(ignore_index=0, label_smoothing=config.label_smoothing)
optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)

Here we implement a linear learning rate scheduler with warmup. The learning rate increases linearly during the warmup phase and then decreases linearly for the rest of the training. We set the warmup to 4% of the total training steps.

In [18]:
# After optimizer definition
def get_linear_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps):
    """Linear warmup then linear decay"""
    def lr_lambda(current_step):
        if current_step < num_warmup_steps:
            return float(current_step) / float(max(1, num_warmup_steps))
        return max(0.0, float(num_training_steps - current_step) /
                   float(max(1, num_training_steps - num_warmup_steps)))

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# Calculate steps
steps_per_epoch = len(train_loader) // config.gradient_accumulation_steps
num_training_steps = steps_per_epoch * config.epochs
num_warmup_steps = int(0.04 * num_training_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps,
    num_training_steps
)

In [19]:
print("Warmup schedule check:")
print(f"Total training steps: {num_training_steps}")
print(f"Warmup steps: {num_warmup_steps}")
print(f"Warmup epochs: {num_warmup_steps / steps_per_epoch:.1f}")

Warmup schedule check:
Total training steps: 99550
Warmup steps: 3982
Warmup epochs: 2.0


Here we implemented gready and beam search translation functions using SentencePiece tokenization. The beam search was supposed to improve translation quality but unfortunately it did not help much in our case.

In [24]:
def translate_sp(model, sentence, src_sp, tgt_sp, device, max_length=64):
    """Translate using SentencePiece"""
    model.eval()

    # Encode
    src_tokens = src_sp.encode(sentence, out_type=int, add_bos=True, add_eos=True)
    src_tokens = src_tokens[:max_length] + [0]*(max_length-len(src_tokens))
    src_tensor = torch.tensor([src_tokens]).to(device)

    # Decode
    tgt_tokens = [2]  # BOS
    with torch.no_grad():
        for _ in range(max_length-1):
            # Pad target to max_length
            tgt_padded = tgt_tokens + [0]*(max_length-len(tgt_tokens))
            tgt_tensor = torch.tensor([tgt_padded]).to(device)

            output = model(src_tensor, tgt_tensor)
            next_token = torch.argmax(output[0, len(tgt_tokens)-1, :]).item()

            if next_token == 3:  # EOS
                break
            tgt_tokens.append(next_token)

    return tgt_sp.decode(tgt_tokens[1:])  # Skip BOS

def translate_sp_beam(model, sentence, src_sp, tgt_sp, device, max_length=64, beam_size=4):
    """Translate using beam search"""
    model.eval()

    # Encode source
    src_tokens = src_sp.encode(sentence, out_type=int, add_bos=True, add_eos=True)
    src_tokens = src_tokens[:max_length] + [0]*(max_length-len(src_tokens))
    src_tensor = torch.tensor([src_tokens]).to(device)

    # Beam search
    from torch.nn.functional import log_softmax

    # Start with BOS
    beams = [([2], 0.0)]  # (tokens, score)

    with torch.no_grad():
        for step in range(max_length - 1):
            new_beams = []

            for tokens, score in beams:
                if tokens[-1] == 3:  # EOS
                    new_beams.append((tokens, score))
                    continue

                # Get predictions
                tgt_padded = tokens + [0]*(max_length-len(tokens))
                tgt_tensor = torch.tensor([tgt_padded]).to(device)
                output = model(src_tensor, tgt_tensor)

                # Get top-k next tokens
                log_probs = log_softmax(output[0, len(tokens)-1, :], dim=0)
                top_probs, top_indices = torch.topk(log_probs, beam_size)

                for prob, idx in zip(top_probs, top_indices):
                    new_tokens = tokens + [idx.item()]
                    new_score = score + prob.item()
                    new_beams.append((new_tokens, new_score))

            # Keep top beam_size
            beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:beam_size]

            # Stop if all beams ended
            if all(tokens[-1] == 3 for tokens, _ in beams):
                break

    # Return best beam
    best_tokens = beams[0][0][1:]  # Skip BOS
    return tgt_sp.decode(best_tokens)

def calculate_val_bleu_sp(model, val_loader, src_sp, tgt_sp, device, max_samples=500):
    """Calculate BLEU with SentencePiece"""
    from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

    model.eval()
    references = []
    hypotheses = []

    with torch.no_grad():
        for batch_idx, (src, tgt) in enumerate(val_loader):
            if batch_idx * val_loader.batch_size >= max_samples:
                break

            src = src.to(device)
            batch_size = src.size(0)

            for i in range(batch_size):
                # Translate
                src_sent = src_sp.decode(src[i].tolist())
                pred_sent = translate_sp_beam(model, src_sent, src_sp, tgt_sp, device)

                # Reference
                ref_sent = tgt_sp.decode([t for t in tgt[i].tolist() if t not in [0, 2, 3]])

                if pred_sent.strip():
                    hypotheses.append(pred_sent.lower().split())
                    references.append([ref_sent.lower().split()])

    if hypotheses:
        smoothing = SmoothingFunction().method1
        return corpus_bleu(references, hypotheses, smoothing_function=smoothing)
    return 0.0

In [21]:
def evaluate(model, dataloader, criterion, device):
    """Evaluate model"""
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for src, tgt in tqdm(dataloader, desc="Evaluating", leave=False):
            src = src.to(device)
            tgt = tgt.to(device)

            tgt_input = tgt[:, :-1]
            tgt_output = tgt[:, 1:]

            output = model(src, tgt_input)
            output = output.reshape(-1, output.size(-1))
            tgt_output = tgt_output.reshape(-1)

            loss = criterion(output, tgt_output)
            total_loss += loss.item()

    return total_loss / len(dataloader)

In [22]:
start_epoch = 0
best_val_loss = float('inf')

In [ ]:
print("\n" + "="*50)
print("Starting training...")
print("="*50)
start_time = datetime.now()

epochs_without_improvement = 0
history = []

for epoch in range(start_epoch, config.epochs):
    print(f"\nEpoch {epoch + 1}/{config.epochs}")

    print(f"Current LR: {optimizer.param_groups[0]['lr']:.6f}")
    print(f"Time elapsed: {datetime.now() - start_time}")

    print(f"Early stopping in {config.early_stopping_patience - epochs_without_improvement} epochs if no improvement (epochs without improvement: {epochs_without_improvement})")

    # Train & Evaluate
    train_loss = train_epoch(model, train_loader, criterion, optimizer, scheduler, device, config)
    val_loss = evaluate(model, val_loader, criterion, device)

    if epoch < 25:  # First 25 epochs
        current_lr = optimizer.param_groups[0]['lr']
        step = epoch * len(train_loader)
        expected_lr = config.learning_rate * min(1.0, step / num_warmup_steps)
        print(f"  Step {step}: LR = {current_lr:.6f} (expected: {expected_lr:.6f})")

    # BLEU every 10 epochs to save time
    if (epoch + 1) % 10 == 0:
        val_bleu = calculate_val_bleu_sp(model, val_loader, pl_sp, en_sp, device, max_samples=500)
        print(f"Validation BLEU: {val_bleu:.4f}")

    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    # Check improvement
    if val_loss < best_val_loss - 0.01:
        best_val_loss = val_loss
        epochs_without_improvement = 0

        # Save best model
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_loss': val_loss
        }, os.path.join(config.checkpoint_dir, 'best_model.pt'))
        print(f"  ✓ Saved best model (val_loss: {val_loss:.4f})")
    else:
        epochs_without_improvement += 1

    # Reduce LR if stuck
    if epochs_without_improvement >= config.lr_reduce_patience:
        current_lr = optimizer.param_groups[0]['lr']
        new_lr = current_lr * config.lr_reduce_factor

        if new_lr > 1e-6:
            for param_group in optimizer.param_groups:
                param_group['lr'] = new_lr
            print(f"  Reduced LR: {current_lr:.6f} → {new_lr:.6f}")
            epochs_without_improvement = 0
        else:
            print("  LR too small, stopping")
            break

    # Early stopping, there's some bug here, but it doesn't matter much for training.
    if epochs_without_improvement >= config.early_stopping_patience:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

    # Sample translations every 5 epochs
    if (epoch + 1) % 5 == 0:
        print("\nSample translations:")
        for test_sent in ["Dzień dobry", "Jak się masz?", "Dziękuję bardzo"]: # Hello, How are you?, Thank you very much
            translation = translate_sp_beam(model, test_sent, pl_sp, en_sp, device)
            print(f"  PL: {test_sent} → EN: {translation}")

    history_point = {
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'val_loss': val_loss,
        'best_val_loss': best_val_loss,
        'epochs_without_improvement': epochs_without_improvement,
        'lr': optimizer.param_groups[0]['lr']
    }
    history.append(history_point)

    # save checkpoint every epoch
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_val_loss': best_val_loss,
        'epochs_without_improvement': epochs_without_improvement,
    }, os.path.join(config.checkpoint_dir, f'checkpoint_epoch_{epoch+1}.pt'))

    # log to file
    with open(os.path.join(config.checkpoint_dir, config.log_file), 'a') as f:
        f.write(json.dumps(history_point) + '\n')

print(f"\nBest val loss: {best_val_loss:.4f}")

with open(os.path.join(config.checkpoint_dir, 'training_history.json'), 'w') as f:
    json.dump(history, f, indent=4)

In [22]:
# load best model
checkpoint = torch.load('checkpoints/best_model.pt')
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

TranslationTransformer(
  (src_embedding): Embedding(16000, 512)
  (tgt_embedding): Embedding(16000, 512)
  (pos_encoder): PositionalEncoding(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-5): 6 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
          )
          (linear1): Linear(in_features=512, out_features=2048, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=2048, out_features=512, bias=True)
          (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.1, inplace=False)
          (dropout2): Dropout(p=0.1, inplace=False)
        )
      )
      (norm): LayerNorm((512,), eps=1e-05, el

In [26]:
test_sentences = [
    "Dzień dobry",# Hello
    "Jak się masz?", # How are you?
    "Dziękuję bardzo", # Thank you very much
    "Jestem szczęśliwy", # I am happy
    "To jest pies", # This is a dog
    "To jest bardzo długie zdanie, które powinno zostać przetłumaczone na angielski" # This is a very long sentence that should be translated into English
]

print("Sample translations from BEST model:")
for pl in test_sentences:
    en = translate_sp_beam(model, pl, pl_sp, en_sp, device)
    print(f"PL: {pl}")
    print(f"EN: {en}\n")

Sample translations from BEST model:
PL: Dzień dobry
EN: Good morning, you're gonna have a good day.

PL: Jak się masz?
EN: How much do you have?

PL: Dziękuję bardzo
EN: Thank you very much for coming in here.

PL: Jestem szczęśliwy
EN: I'm happy to be happy.

PL: To jest pies
EN: This is a dog's dog, and it's a dog.

PL: To jest bardzo długie zdanie, które powinno zostać przetłumaczone na angielski
EN: This is a very long term, so it should be a long time to stay in English.



The translation quality is definitly not bad, the key words are translated correctly, but it is adding a lot of unnecessary words. Although the long sentence did not translate correctly.

In [26]:
# This should be MUCH higher than test BLEU
val_bleu = calculate_val_bleu_sp(
    model, val_loader, pl_sp, en_sp, device, max_samples=1000
)
print(f"Validation BLEU: {val_bleu:.4f}")

C:\Projects\BUAS\Y2\block-a\fae2-nlpr-group-group-9-1\.venv\Lib\site-packages\torch\nn\functional.py:6041: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


Validation BLEU: 0.0545


Low validation BLEU indicates that the model did not learn to generalize well.

In [25]:
# Test on kinga.csv
df_test = pd.read_csv('../../data/kinga.csv')

translations = []
for sent in df_test['Sentence'].tolist():
    translation = translate_sp_beam(model, sent, pl_sp, en_sp, device)
    translations.append(translation)

# Calculate BLEU
from nltk.translate.bleu_score import corpus_bleu
references = [[ref.lower().split()] for ref in df_test['Translation'].tolist()]
hypotheses = [hyp.lower().split() for hyp in translations]
bleu = corpus_bleu(references, hypotheses)
print(f"Test BLEU: {bleu:.4f}")

C:\Projects\BUAS\Y2\block-a\fae2-nlpr-group-group-9-1\.venv\Lib\site-packages\torch\nn\functional.py:6041: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


Test BLEU: 0.0460


In [27]:
df_test["Translation_Predicted"] = translations
df_test.to_csv('kinga_translations.csv', index=False)

In [28]:
df_test.head()

,Start Time,End Time,Sentence,Translation,Emotion_fine,Emotion_core,Intensity,Translation_Predicted
0,1/1/1900 0:00,1/1/1900 0:00,To jest MasterChef.,This is MasterChef.,anticipation,neutral,neutral,"This is the MCCCCC, which is in the middle of ..."
1,1/1/1900 0:00,1/1/1900 0:00,Szansę na tytuł najlepszego kucharza w Polsce ...,Only 12 people still have a chance at the titl...,anticipation,neutral,mild,The new cook has only 12 people in Poland.
2,1/1/1900 0:00,1/1/1900 0:00,Oto oni.,Here they are.,anticipation,surprise,mild,"Here they go, and they're going to be in the m..."
3,1/1/1900 0:00,1/1/1900 0:00,"Jestem dinozaurem, który chce walczyć, który s...","I am a dinosaur who wants to fight, who will n...",determination,happiness,intense,"I'm the one who's trying to fight for a fight,..."
4,1/1/1900 0:00,1/1/1900 0:00,Ten program daje mi ogromną siłę.,This program gives me tremendous strength.,empowerment,happiness,intense,This program gives me a big deal.


The translations are not usable. The model seems to struggle with generalization. The more in-depth error analysis will be provided in the report.

In [ ]:
# clean output log from all "Training:" and "Evaluating:" lines
with open('checkpoints/output.log', 'r') as f:
    lines = f.readlines()
with open('checkpoints/output_clean.log', 'w') as f:
    for line in lines:
        if not line.startswith('Training:') and not line.startswith('Evaluating:'):
            f.write(line)